# Milestone 5 — Phase 3: Probability Calibration and Threshold Optimization

This notebook compares uncalibrated, sigmoid-calibrated and isotonic-calibrated probabilities for the Phase 2A finalists. It uses nested stratified cross-validation on the original training partition only. The held-out test partition is not passed to any experiment. The primary threshold objective is F1 as a practical balance between precision and recall.

**Candidates:** XGBoost unweighted, XGBoost balanced and Random Forest balanced.


## 1. Environment and dataset

Run from the repository root or Google Colab. Upload the original IBM Telco CSV if it is not already in `data/raw/`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

EXPECTED_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
current_dir = Path.cwd()
PROJECT_ROOT = next((p for p in (current_dir, *current_dir.parents) if (p / "src").is_dir()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/content/customer-churn-prediction")
    if not (PROJECT_ROOT / "src").is_dir():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/adhishmodi/customer-churn-prediction.git", str(PROJECT_ROOT)], check=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "raw" / EXPECTED_CSV
if not DATA_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    if EXPECTED_CSV not in uploaded:
        raise FileNotFoundError(f"Upload the original file named {EXPECTED_CSV}")
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    DATA_PATH.write_bytes(uploaded[EXPECTED_CSV])
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")


## 2. Reconstruct the original training partition

The original 80/20 stratified split is recreated with seed 42. The holdout rows are immediately discarded and never passed into the experiment function.


In [ ]:
from src.preprocess import clean_data, prepare_features_target, split_data
from src.train import get_imbalance_experiment_models
from src.calibration import evaluate_calibration_threshold_cv

raw_df = pd.read_csv(DATA_PATH)
cleaned_df = clean_data(raw_df)
X_all, y_all = prepare_features_target(cleaned_df)
X_train, discarded_features, y_train, discarded_labels = split_data(X_all, y_all)
del raw_df, cleaned_df, X_all, y_all, discarded_features, discarded_labels
print(f"Training rows: {len(X_train):,}")
print(f"Training churn prevalence: {y_train.mean():.2%}")


## 3. Run nested calibration and threshold experiments

For each outer fold, inner out-of-fold probabilities from the outer-training partition fit the calibration mapping and select the F1 threshold. A fresh model is fitted on the complete outer-training partition and evaluated on the untouched outer-validation fold. This evaluates the complete calibration-plus-threshold procedure without using the project's held-out test data.


In [ ]:
all_candidates = get_imbalance_experiment_models()
candidate_names = [
    "XGBoost (unweighted)",
    "XGBoost (balanced)",
    "Random Forest (balanced)",
]
candidates = {name: all_candidates[name] for name in candidate_names}
results = evaluate_calibration_threshold_cv(
    X_train,
    y_train,
    models=candidates,
    outer_splits=5,
    inner_splits=5,
    random_state=42,
)
fold_scores = results["fold_scores"]
summary = results["summary"]
validation_predictions = results["validation_predictions"]
assert results["configuration"]["test_set_used"] is False
display(summary.sort_values(["model", "f1_mean"], ascending=[True, False]))
print(results["configuration"])


## 4. Calibration diagnostics

Reliability curves use only outer-fold validation predictions. Lower Brier score and log loss indicate better probabilistic accuracy; ROC-AUC and Average Precision describe ranking and are not threshold-dependent.


In [ ]:
for model_name in candidate_names:
    fig, ax = plt.subplots(figsize=(7, 5))
    subset = validation_predictions[validation_predictions["model"] == model_name]
    for method in ("none", "sigmoid", "isotonic"):
        method_rows = subset[subset["calibration"] == method]
        observed, predicted = calibration_curve(
            method_rows["y_true"],
            method_rows["calibrated_probability"],
            n_bins=10,
            strategy="quantile",
        )
        ax.plot(predicted, observed, marker="o", label=method)
    ax.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    ax.set(title=f"Calibration curve — {model_name}", xlabel="Mean predicted probability", ylabel="Observed churn frequency")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


## 5. Precision-recall balance and threshold stability

Compare the outer-fold precision, recall and F1 results. The standard deviation of fold-selected thresholds is descriptive fold variability, not a confidence interval.


In [ ]:
metric_columns = ["precision_mean", "recall_mean", "f1_mean", "threshold_mean", "threshold_std", "brier_score_mean", "log_loss_mean"]
display(summary[["model", "calibration", *metric_columns]].sort_values("f1_mean", ascending=False))
fig, ax = plt.subplots(figsize=(10, 5))
plot_rows = summary.sort_values("f1_mean")
labels = plot_rows["model"] + " — " + plot_rows["calibration"]
ax.barh(labels, plot_rows["f1_mean"], xerr=plot_rows["f1_std"], capsize=3)
ax.set(xlabel="Outer-fold F1 (mean ± standard deviation)", title="Calibration and threshold procedure comparison", xlim=(0, 1))
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## 6. Save reproducible reports

Generated CSVs are stored under `reports/`, which is intentionally ignored by Git. Download these files from Colab after execution and retain them with the project experiment records.


In [ ]:
reports_dir = PROJECT_ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
outputs = {
    "phase_3_calibration_threshold_fold_scores.csv": fold_scores,
    "phase_3_calibration_threshold_summary.csv": summary,
    "phase_3_calibration_threshold_validation_predictions.csv": validation_predictions,
}
for filename, frame in outputs.items():
    path = reports_dir / filename
    frame.to_csv(path, index=False)
    print(f"Saved: {path}")


## 7. Interpretation checklist

1. Which candidate and calibration method provide the strongest outer-fold F1 while maintaining a reasonable precision-recall balance?
2. Does calibration improve Brier score and log loss compared with uncalibrated probabilities?
3. Are selected thresholds reasonably stable across outer folds?
4. Are ROC-AUC and Average Precision broadly preserved after calibration?
5. Do conclusions remain consistent across all outer folds?

Do not select a final production model from one metric alone. These results are training-only nested-CV estimates. Phase 4 will freeze the candidate and threshold using the agreed validation protocol, then evaluate the held-out test partition once.
